In [ ]:
!pip install torch torchvision -q
print("✅ Ready")

✅ Ready


In [ ]:
from google.colab import files
import zipfile, shutil, glob, os
from pathlib import Path

BASE = Path("/content/neuroai")
for folder in [
    "data/train/ASD",
    "data/train/NonASD",
    "data/val/ASD",
    "data/val/NonASD",
    "models"
]:
    (BASE/folder).mkdir(
        parents=True, exist_ok=True
    )

print("Upload mri_new_dataset_sem7.zip")
uploaded = files.upload()

# ── Fix: find the actual filename ─────────────
zip_filename = list(uploaded.keys())[0]
print(f"Uploaded as: '{zip_filename}'")

print("Extracting...")
with zipfile.ZipFile(zip_filename) as z:
    z.extractall("/content/raw/")

print("Moving images...")
# Move images to correct folders
for f in glob.glob(
    "/content/raw/**/autism/*.png",
    recursive=True
):
    split = "train" if "train" in f else "val"
    shutil.copy(
        f, BASE/f"data/{split}/ASD/"
    )

for f in glob.glob(
    "/content/raw/**/control/*.png",
    recursive=True
):
    split = "train" if "train" in f else "val"
    shutil.copy(
        f, BASE/f"data/{split}/NonASD/"
    )

# Verify counts
print("\n✅ Dataset loaded:")
for split in ['train', 'val']:
    for cls in ['ASD', 'NonASD']:
        count = len(list(
            (BASE/f"data/{split}/{cls}"
             ).glob("*.png")
        ))
        print(f"  {split}/{cls}: {count}")

print("\n✅ Ready for training!")

Upload mri_new_dataset_sem7.zip


Saving mri_new dataset sem7.zip to mri_new dataset sem7 (1).zip
Uploaded as: 'mri_new dataset sem7 (1).zip'
Extracting...
Moving images...

✅ Dataset loaded:
  train/ASD: 3884
  train/NonASD: 4647
  val/ASD: 972
  val/NonASD: 1162

✅ Ready for training!


In [ ]:
import os
from pathlib import Path

BASE = Path("/content/neuroai")

print("=== CHECKING /content/raw/ STRUCTURE ===")
for root, dirs, files in os.walk('/content/raw'):
    level = root.replace(
        '/content/raw', ''
    ).count(os.sep)
    indent = ' ' * 2 * level
    basename = os.path.basename(root)
    print(f'{indent}{basename}/')
    if files:
        subindent = ' ' * 2 * (level+1)
        png_count = len(
            [f for f in files
             if f.endswith('.png')]
        )
        print(f'{subindent}{png_count} '
              f'PNG files')
    if level >= 4:
        break

print("\n=== CHECKING neuroai FOLDERS ===")
for split in ['train', 'val']:
    for cls in ['ASD', 'NonASD']:
        path = BASE/f"data/{split}/{cls}"
        count = len(list(path.glob("*.png")))
        print(f"  {split}/{cls}: {count} images")

=== CHECKING /content/raw/ STRUCTURE ===
raw/
  mri_new dataset sem7/
    val/
      control/
        1162 PNG files
      autism/
        972 PNG files
    train/
      control/
        4647 PNG files
      autism/
        3884 PNG files

=== CHECKING neuroai FOLDERS ===
  train/ASD: 3884 images
  train/NonASD: 4647 images
  val/ASD: 972 images
  val/NonASD: 1162 images


In [ ]:
# COMPLETE TRAINING CELL — Run this now
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)
from PIL import Image
from pathlib import Path
import pickle, time, random
from sklearn.metrics import (
    roc_auc_score,
    classification_report
)

BASE = Path("/content/neuroai")
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Device: {device}")

# ── Dataset Class ──────────────────────────────
class MRIDataset(Dataset):
    def __init__(self, asd_dir, non_dir,
                 transform=None):
        self.files = []
        self.labels = []
        self.transform = transform

        for f in Path(asd_dir).glob("*.png"):
            self.files.append(str(f))
            self.labels.append(1)
        for f in Path(non_dir).glob("*.png"):
            self.files.append(str(f))
            self.labels.append(0)

        # Shuffle
        temp = list(zip(
            self.files, self.labels
        ))
        random.shuffle(temp)
        self.files = [x[0] for x in temp]
        self.labels = [x[1] for x in temp]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(
            self.files[idx]
        ).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# ── Transforms ─────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# ── Load Dataset ───────────────────────────────
print("\nLoading dataset...")
train_ds = MRIDataset(
    BASE/"data/train/ASD",
    BASE/"data/train/NonASD",
    transform=train_transform
)
val_ds = MRIDataset(
    BASE/"data/val/ASD",
    BASE/"data/val/NonASD",
    transform=val_transform
)

train_loader = DataLoader(
    train_ds, batch_size=32,
    shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_ds, batch_size=32,
    shuffle=False, num_workers=2
)

print(f"✅ Train: {len(train_ds):,} images")
print(f"✅ Val:   {len(val_ds):,} images")
print(f"✅ No augmentation needed!")

# ── Training Function ──────────────────────────
def train_model(model, model_name,
                save_path, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=0.0001,
        weight_decay=1e-3
    )
    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min',
            patience=3, factor=0.5
        )
    )

    best_acc = 0
    best_auc = 0
    best_epoch = 0
    patience_counter = 0
    EARLY_STOP = 7

    print(f"\n{'='*55}")
    print(f"Training {model_name}")
    print(f"{'='*55}")
    start = time.time()

    for epoch in range(epochs):
        # Train phase
        model.train()
        tr_loss = 0
        tr_correct = 0
        tr_total = 0

        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            optimizer.step()

            tr_loss += loss.item()
            _, pred = torch.max(out, 1)
            tr_total += labels.size(0)
            tr_correct += (
                pred == labels
            ).sum().item()

        # Val phase
        model.eval()
        v_loss = 0
        v_preds = []
        v_labels = []
        v_probs = []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)

                out = model(imgs)
                loss = criterion(out, labels)
                v_loss += loss.item()

                probs = torch.softmax(
                    out, dim=1
                )
                _, pred = torch.max(out, 1)

                v_preds.extend(
                    pred.cpu().numpy()
                )
                v_labels.extend(
                    labels.cpu().numpy()
                )
                v_probs.extend(
                    probs[:, 1].cpu().numpy()
                )

        # Calculate metrics
        tr_acc = tr_correct / tr_total
        v_acc = sum(
            p == l for p, l in
            zip(v_preds, v_labels)
        ) / len(v_labels)
        avg_tr = tr_loss / len(train_loader)
        avg_v = v_loss / len(val_loader)

        try:
            auc = roc_auc_score(
                v_labels, v_probs
            )
        except:
            auc = 0.5

        scheduler.step(avg_v)

        if v_acc > best_acc:
            best_acc = v_acc
            best_auc = auc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(
                model.state_dict(),
                save_path
            )
            star = " ⭐ BEST"
        else:
            patience_counter += 1
            star = ""

        elapsed = (time.time()-start)/60
        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train:{tr_acc:.3f} "
            f"Val:{v_acc:.3f} | "
            f"Loss:{avg_tr:.4f}→"
            f"{avg_v:.4f} | "
            f"AUC:{auc:.3f} | "
            f"⏱{elapsed:.1f}m{star}"
        )

        if patience_counter >= EARLY_STOP:
            print(
                f"\n⏹ Early stop "
                f"at epoch {epoch+1}"
            )
            break

    print(f"\n✅ {model_name} complete!")
    print(f"   Accuracy: {best_acc*100:.1f}%")
    print(f"   AUC:      {best_auc:.4f}")
    print(f"   Epoch:    {best_epoch}")
    print(f"   Val size: {len(val_ds):,}")

    return best_acc, best_auc, best_epoch

# ══════════════════════════════════════════════
# TRAIN RESNET18
# ══════════════════════════════════════════════
print("\n🔵 Building ResNet18...")
resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)
for param in list(
    resnet.parameters()
)[:-20]:
    param.requires_grad = False

resnet.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Dropout(0.3),
    nn.Linear(256, 2)
)
resnet = resnet.to(device)
print("✅ ResNet18 ready")

resnet_acc, resnet_auc, resnet_ep = (
    train_model(
        resnet, "ResNet18",
        BASE/"models/resnet18_new.pth",
        epochs=20
    )
)

# ══════════════════════════════════════════════
# TRAIN EFFICIENTNET
# ══════════════════════════════════════════════
print("\n🟢 Building EfficientNet-B0...")
effnet = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)
for param in list(
    effnet.parameters()
)[:-20]:
    param.requires_grad = False

effnet.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2)
)
effnet = effnet.to(device)
print("✅ EfficientNet ready")

effnet_acc, effnet_auc, effnet_ep = (
    train_model(
        effnet, "EfficientNet-B0",
        BASE/"models/efficientnet_new.pth",
        epochs=20
    )
)

# ══════════════════════════════════════════════
# CREATE + EVALUATE ENSEMBLE
# ══════════════════════════════════════════════
print("\n🔴 Creating MRI Ensemble...")

resnet.load_state_dict(torch.load(
    BASE/"models/resnet18_new.pth",
    map_location=device
))
effnet.load_state_dict(torch.load(
    BASE/"models/efficientnet_new.pth",
    map_location=device
))
resnet.eval()
effnet.eval()

ens_preds = []
ens_labels = []
r_probs = []
e_probs = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)

        pr = torch.softmax(
            resnet(imgs), dim=1
        )
        pe = torch.softmax(
            effnet(imgs), dim=1
        )
        ens = 0.6*pr + 0.4*pe
        _, pred = torch.max(ens, 1)

        ens_preds.extend(
            pred.cpu().numpy()
        )
        ens_labels.extend(
            labels.cpu().numpy()
        )
        r_probs.extend(
            pr[:, 1].cpu().numpy()
        )
        e_probs.extend(
            pe[:, 1].cpu().numpy()
        )

ens_final_probs = [
    0.6*r + 0.4*e
    for r, e in zip(r_probs, e_probs)
]
ens_acc = sum(
    p == l for p, l
    in zip(ens_preds, ens_labels)
) / len(ens_labels)
ens_auc = roc_auc_score(
    ens_labels, ens_final_probs
)

# ── Print final comparison ─────────────────────
print(f"\n{'='*55}")
print(f"FINAL MRI MODEL COMPARISON")
print(f"Validated on {len(val_ds):,} images")
print(f"{'='*55}")
print(
    f"{'Model':<32} "
    f"{'Acc':>8} {'AUC':>8}"
)
print(f"{'-'*50}")
print(
    f"{'PCA+VotingClassifier (old)':<32} "
    f"{'65.0%':>8} {'0.720':>8}"
)
print(
    f"{'ResNet18':<32} "
    f"{resnet_acc*100:>7.1f}% "
    f"{resnet_auc:>8.4f}"
)
print(
    f"{'EfficientNet-B0':<32} "
    f"{effnet_acc*100:>7.1f}% "
    f"{effnet_auc:>8.4f}"
)
print(
    f"{'ResNet+EfficientNet Ensemble':<32} "
    f"{ens_acc*100:>7.1f}% "
    f"{ens_auc:>8.4f}"
)
print(f"{'='*55}")

# ── Save everything ────────────────────────────
import pandas as pd

mri_results = pd.DataFrame([
    {
        'model': 'PCA + VotingClassifier',
        'task': 'mri_binary',
        'test_acc': 0.650,
        'f1': 0.643,
        'auc': 0.720,
        'val_samples': 41,
        'approach': 'Traditional ML',
        'semester': 'Previous'
    },
    {
        'model': 'ResNet18',
        'task': 'mri_binary',
        'test_acc': round(resnet_acc, 4),
        'f1': round(resnet_acc, 4),
        'auc': round(resnet_auc, 4),
        'val_samples': len(val_ds),
        'approach': 'Deep Learning',
        'semester': 'Current'
    },
    {
        'model': 'EfficientNet-B0',
        'task': 'mri_binary',
        'test_acc': round(effnet_acc, 4),
        'f1': round(effnet_acc, 4),
        'auc': round(effnet_auc, 4),
        'val_samples': len(val_ds),
        'approach': 'Deep Learning',
        'semester': 'Current'
    },
    {
        'model': 'ResNet18+EfficientNet',
        'task': 'mri_binary',
        'test_acc': round(ens_acc, 4),
        'f1': round(ens_acc, 4),
        'auc': round(ens_auc, 4),
        'val_samples': len(val_ds),
        'approach': 'Deep Learning Ensemble',
        'semester': 'Current'
    }
])
mri_results.to_csv(
    BASE/"models/mri_results.csv",
    index=False
)

# Save ensemble bundle
bundle = {
    'resnet_state': resnet.state_dict(),
    'effnet_state': effnet.state_dict(),
    'resnet_weight': 0.6,
    'effnet_weight': 0.4,
    'resnet_accuracy': resnet_acc,
    'effnet_accuracy': effnet_acc,
    'ensemble_accuracy': ens_acc,
    'ensemble_auc': ens_auc,
    'img_size': 224,
    'classes': ['NonASD', 'ASD'],
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
    'model_type': 'ensemble_resnet18_efficientnet',
    'dataset': 'ABIDE 10665 images',
    'val_samples': len(val_ds),
    'train_samples': len(train_ds)
}

with open(
    BASE/"models/mri_ensemble_new.pkl", 'wb'
) as f:
    pickle.dump(bundle, f)

print(f"\n✅ All files saved!")

# ── Download everything ────────────────────────
from google.colab import files

print("\nDownloading all files now...")
files.download(
    str(BASE/"models/mri_ensemble_new.pkl")
)
files.download(
    str(BASE/"models/resnet18_new.pth")
)
files.download(
    str(BASE/"models/efficientnet_new.pth")
)
files.download(
    str(BASE/"models/mri_results.csv")
)
print("\n✅ ALL DOWNLOADED!")
print("Copy these to Desktop/NeuroAI/models/")
print("Colab work is DONE FOREVER after this!")

Device: cuda

Loading dataset...
✅ Train: 8,531 images
✅ Val:   2,134 images
✅ No augmentation needed!

🔵 Building ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 53.6MB/s]


✅ ResNet18 ready

Training ResNet18
Epoch 01/20 | Train:0.580 Val:0.645 | Loss:0.7075→0.6203 | AUC:0.700 | ⏱0.8m ⭐ BEST
Epoch 02/20 | Train:0.643 Val:0.647 | Loss:0.6351→0.6325 | AUC:0.718 | ⏱1.5m ⭐ BEST
Epoch 03/20 | Train:0.675 Val:0.696 | Loss:0.5969→0.5844 | AUC:0.771 | ⏱2.2m ⭐ BEST
Epoch 04/20 | Train:0.712 Val:0.691 | Loss:0.5589→0.5930 | AUC:0.776 | ⏱3.0m
Epoch 05/20 | Train:0.736 Val:0.726 | Loss:0.5195→0.5623 | AUC:0.799 | ⏱3.7m ⭐ BEST
Epoch 06/20 | Train:0.761 Val:0.744 | Loss:0.4847→0.5025 | AUC:0.826 | ⏱4.4m ⭐ BEST
Epoch 07/20 | Train:0.791 Val:0.737 | Loss:0.4442→0.5509 | AUC:0.820 | ⏱5.2m
Epoch 08/20 | Train:0.808 Val:0.758 | Loss:0.4112→0.4905 | AUC:0.844 | ⏱5.9m ⭐ BEST
Epoch 09/20 | Train:0.824 Val:0.741 | Loss:0.3907→0.5346 | AUC:0.840 | ⏱6.6m
Epoch 10/20 | Train:0.833 Val:0.770 | Loss:0.3702→0.4901 | AUC:0.863 | ⏱7.4m ⭐ BEST
Epoch 11/20 | Train:0.849 Val:0.776 | Loss:0.3413→0.5104 | AUC:0.862 | ⏱8.1m ⭐ BEST
Epoch 12/20 | Train:0.863 Val:0.787 | Loss:0.3160→0.4934 | AU

100%|██████████| 20.5M/20.5M [00:00<00:00, 135MB/s] 


✅ EfficientNet ready

Training EfficientNet-B0
Epoch 01/20 | Train:0.581 Val:0.586 | Loss:0.6735→0.6592 | AUC:0.627 | ⏱0.8m ⭐ BEST
Epoch 02/20 | Train:0.615 Val:0.604 | Loss:0.6471→0.6442 | AUC:0.660 | ⏱1.5m ⭐ BEST
Epoch 03/20 | Train:0.633 Val:0.625 | Loss:0.6344→0.6371 | AUC:0.678 | ⏱2.3m ⭐ BEST
Epoch 04/20 | Train:0.641 Val:0.615 | Loss:0.6230→0.6308 | AUC:0.677 | ⏱3.0m
Epoch 05/20 | Train:0.659 Val:0.634 | Loss:0.6120→0.6237 | AUC:0.700 | ⏱3.8m ⭐ BEST
Epoch 06/20 | Train:0.664 Val:0.650 | Loss:0.6031→0.6081 | AUC:0.714 | ⏱4.5m ⭐ BEST
Epoch 07/20 | Train:0.673 Val:0.649 | Loss:0.5893→0.6197 | AUC:0.711 | ⏱5.3m
Epoch 08/20 | Train:0.678 Val:0.663 | Loss:0.5894→0.5933 | AUC:0.734 | ⏱6.1m ⭐ BEST
Epoch 09/20 | Train:0.689 Val:0.673 | Loss:0.5748→0.5912 | AUC:0.738 | ⏱6.8m ⭐ BEST
Epoch 10/20 | Train:0.702 Val:0.659 | Loss:0.5679→0.6020 | AUC:0.734 | ⏱7.6m
Epoch 11/20 | Train:0.696 Val:0.680 | Loss:0.5628→0.5796 | AUC:0.751 | ⏱8.4m ⭐ BEST
Epoch 12/20 | Train:0.717 Val:0.679 | Loss:0.5502→

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ALL DOWNLOADED!
Copy these to Desktop/NeuroAI/models/
Colab work is DONE FOREVER after this!
